# Auditoría independiente MER v2 — junio, julio, agosto o septiembre 2026

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/MercadoPublico-Inteligencia/blob/main/colab/Auditoria_Integridad_MER_v2_Colab.ipynb)

**Solo lectura.** Configura `PERIODO` en la primera celda de código (por defecto `2026-08`). Selecciona exclusivamente el Sheet registrado para ese período y verifica las seis tablas, PK, relaciones y cobertura de OC sin escribir datos. Descarga un informe JSON identificado por mes.

In [ ]:
!pip -q install gspread google-auth
from google.colab import auth, files
from google.auth import default
import gspread, collections, json, datetime, time, pathlib
auth.authenticate_user()
cred,_=default(scopes=['https://www.googleapis.com/auth/spreadsheets.readonly'])
gc=gspread.authorize(cred)
PERIODO='2026-08'  # cambiar a 2026-07 o 2026-06 al avanzar
DESTINOS={
 '2026-06':'137pknbQIBbm5WAYvFh5bXKTGX9uBVhBYleTwxO6x7PI',
 '2026-07':'1LcVUrWn99deDkgphD8cAn2n4jLxl-HZyzNm0XGffC5w',
 '2026-08':'1TY41r2eriz3wvvLvHdYxHVrbqk94Lsp1kFtDtxxcXEs',
 '2026-09':'1lnMVUZ_pn_HkwEvI90yYz5GOMSYzuX-FbcbMR71-oDA'
}
assert PERIODO in DESTINOS, 'Período no registrado'
ID=DESTINOS[PERIODO]
print('AUDITORÍA SOLO LECTURA:',PERIODO,'Sheet:',ID)
book=gc.open_by_key(ID)
ESPERADOS_POR_PERIODO={
 '2026-08':{'Procesos':83937,'Proveedores':30564,'Ofertas':432111,'OrdenesCompra':153221,'VinculosCOT_OC':68592,'Resultados':65945},
 '2026-09':{'Procesos':79376,'Proveedores':30280,'Ofertas':398018,'OrdenesCompra':147966,'VinculosCOT_OC':50041,'Resultados':48637}
}
SCHEMA={
'Procesos':('codigo',['codigo','nombre','estado','convocatoria','fecha_publicacion','fecha_cierre','monto_disponible_clp','organismo_comprador','region','total_ofertas_recibidas','motivo_seleccion','url_detalle','fecha_sincronizacion','categoria','subcategoria','modalidad'],0),
'Proveedores':('id_proveedor',['id_proveedor','rut_canonico','razon_social_canonica','nombre_fantasia','fuente_principal','estado_identidad'],0),
'Ofertas':('id_cotizacion',['id_cotizacion','codigo','id_proveedor','proveedor','monto_ofertado_clp','fecha_oferta','fuente','seleccionado','rut_proveedor_cot'],0),
'OrdenesCompra':('codigo_oc',['codigo_oc','codigo_proceso','id_proveedor','fecha_creacion','fecha_aceptacion','estado','proveedor','monto_total_oc_clp','tipo_oc','cantidad_lineas','total_lineas_neto','rut_sucursal_oc','codigo_proveedor_oc'],0),
'VinculosCOT_OC':('id_vinculo',['id_vinculo','codigo_cot','codigo_oc','origen_vinculo','estado_oc_historico','fecha_aceptacion_oc','relacion_confiable'],0),
'Resultados':('codigo',['codigo','id_cotizacion_ganadora','adjudicatario','monto_adjudicado_clp','fecha_resultado','motivo_seleccion','fuente','validado','monto_oferta_seleccionada_clp','estado_verificacion_oc'],0)}
print('Conectado a MER v2; sin permisos de escritura.')
for n,(pk,cab,_) in list(SCHEMA.items()):
 SCHEMA[n]=(pk,cab,ESPERADOS_POR_PERIODO.get(PERIODO,{}).get(n))
print('Conteos esperados:',{n:x[2] for n,x in SCHEMA.items()})

In [ ]:
def leer(nombre,lote=3000):
 pk,cab,esperado=SCHEMA[nombre];ws=book.worksheet(nombre)
 assert ws.row_values(1)==cab, f'Encabezados incorrectos: {nombre}'
 filas=[]
 for inicio in range(2,ws.row_count+1,lote):
  fin=min(ws.row_count,inicio+lote-1)
  try:
   batch=ws.get(f'A{inicio}:{gspread.utils.rowcol_to_a1(1,len(cab)).rstrip("1")}{fin}',value_render_option='UNFORMATTED_VALUE')
  except Exception as e:
   raise RuntimeError(f'Error leyendo {nombre} filas {inicio}-{fin}: {e}')
  for r in batch:
   if not any(str(v).strip() for v in r): continue
   r=[str(v).strip() if v is not None else '' for v in r]
   filas.append(dict(zip(cab,(r+['']*len(cab))[:len(cab)])))
  if inicio%30000<lote: print(nombre,'recuperadas:',len(filas))
 assert esperado is None or len(filas)==esperado, f'Conteo {nombre} = {len(filas)}, esperado {esperado}'
 return filas
datos={n:leer(n) for n in SCHEMA}
print('Se recuperaron todas las filas sin escrituras.')

In [ ]:
rep={'fecha_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'base':f'MER v2 {PERIODO}; solo lectura','periodo':PERIODO,'spreadsheet_id':ID,'controles':{},'relaciones':{},'observaciones':[]}
indices={}
for n,rs in datos.items():
 pk,cab,esperado=SCHEMA[n]
 claves=[r[pk] for r in rs]
 indices[n]=set(claves)
 rep['controles'][n]={'filas':len(rs),'pk_vacia':sum(not x for x in claves),
 'pk_duplicadas':len(claves)-len(indices[n]),'campos_extra_o_ausentes':sum(set(r)!=set(cab) for r in rs)}
P=indices['Procesos'];V=indices['Proveedores'];O=indices['Ofertas'];OC=indices['OrdenesCompra']
Omap={r['id_cotizacion']:r for r in datos['Ofertas']}
def contar(tabla,campo,referencias,permitir_vacio=False):
 return sum((bool(r[campo]) or not permitir_vacio) and r[campo] not in referencias for r in datos[tabla])
rep['relaciones']={
 'ofertas_sin_proceso':contar('Ofertas','codigo',P),
 'ofertas_sin_proveedor':contar('Ofertas','id_proveedor',V,True),
 'resultados_sin_proceso':contar('Resultados','codigo',P),
 'resultados_sin_oferta':contar('Resultados','id_cotizacion_ganadora',O),
 'resultados_oferta_otro_proceso':sum(r['id_cotizacion_ganadora'] in Omap and Omap[r['id_cotizacion_ganadora']]['codigo']!=r['codigo'] for r in datos['Resultados']),
 'oc_con_proveedor_inexistente':contar('OrdenesCompra','id_proveedor',V,True),
 'vinculos_sin_proceso':contar('VinculosCOT_OC','codigo_cot',P),
 'vinculos_oc_fuera_de_corte':contar('VinculosCOT_OC','codigo_oc',OC),
 'vinculos_par_cot_oc_repetido':len(datos['VinculosCOT_OC'])-len({(r['codigo_cot'],r['codigo_oc']) for r in datos['VinculosCOT_OC']})
}
rep['estados_resultados']=dict(collections.Counter(r['estado_verificacion_oc'] for r in datos['Resultados']))
rep['estados_proveedores']=dict(collections.Counter(r['estado_identidad'] for r in datos['Proveedores']))
rep['observaciones'].append('Vínculos OC fuera del corte no son errores, conforme a decisión metodológica')
rep['estado']='AUDITORIA_COMPLETADA_CON_OBSERVACIONES' if rep['relaciones']['vinculos_oc_fuera_de_corte'] else 'AUDITORIA_COMPLETADA'
print(json.dumps(rep,ensure_ascii=False,indent=2))

In [ ]:
salida=f'/content/auditoria_integridad_mer_v2_{PERIODO}.json'
with open(salida,'w',encoding='utf-8') as f:json.dump(rep,f,ensure_ascii=False,indent=2)
files.download(salida)
print('Auditoría de solo lectura finalizada. Descarga el JSON y compártelo para documentar el cierre.')